# Performance Tuning — Lift TF-IDF + LR from 0.79 → higher micro-F1

Runs 7 model variants end-to-end, logs metrics after each, produces an ablation table and a per-clause F1 chart. CPU-only — designed to finish in ~5 minutes on a Mac.

| Variant | What it adds |
|---------|--------------|
| V1 | Baseline: TF-IDF (1–2 word grams, 50k vocab) + LR, C=1, global threshold |
| V2 | + word n-grams (1–3) + char n-grams (3–5), 150k total features |
| V3 | + per-clause `C` grid search {0.1, 1, 10} (guarded by min-class ≥ 10) |
| V4 | + probability calibration (Platt scaling) |
| V5 | + per-clause thresholds (tuned on validation) |
| V6 | + MiniLM **late-fusion** ensemble (probability averaging, weight_a swept on val) |
| V7 | + MiniLM **early-fusion** (TF-IDF ⊕ MiniLM concatenated features, joint LR) |

**Outputs:** `figures/ablation_table.csv`, `figures/per_clause_f1_chart.png`, `checkpoints/tfidf_lr_v2_artifacts.joblib`

## Section 0 — Setup

In [11]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))

from data_loading import load_cuad
from preprocessing import (
    filter_clauses, build_clause_mappings,
    build_contract_records, split_contract_records,
)
from training import (
    train_tfidf_lr,           # V1
    train_tfidf_lr_v2,        # V2, V3, V4
    train_minilm_lr,          # V6 component
    ensemble_artifacts,       # V6 ensemble
    train_hybrid_features_lr, # V7 early-fusion
)
from evaluation import (
    tune_per_clause_thresholds,
    compute_per_clause_metrics,
    compute_aggregate_metrics,
)

Path('figures').mkdir(exist_ok=True)
Path('checkpoints').mkdir(exist_ok=True)
print('Imports OK.')

ImportError: cannot import name 'train_hybrid_features_lr' from 'training' (/Users/saiashwin/BT5153/.claude/worktrees/crazy-tharp-7c055d/training.py)

## Section 1 — Load Data & Build Splits

In [ ]:
cuad_df = load_cuad('data/cuad')
cuad_filtered, _ = filter_clauses(cuad_df, min_positives=20)
clause_to_id, id_to_clause = build_clause_mappings(cuad_filtered)

records = build_contract_records(cuad_filtered)
train_recs, val_recs, test_recs = split_contract_records(records, seed=42)

train_titles = {r['contract_title'] for r in train_recs}
val_titles   = {r['contract_title'] for r in val_recs}

train_df = cuad_filtered[cuad_filtered['contract_title'].isin(train_titles)].copy()
val_df   = cuad_filtered[cuad_filtered['contract_title'].isin(val_titles)].copy()

print(f'{len(id_to_clause)} clauses · {len(train_titles)} train / {len(val_titles)} val contracts')

CUAD loaded: 510 contracts, 41 clause types, 20,910 rows, positive rate 32.05%
Excluded 3 clause types (below min_positives=20):
  Source Code Escrow: 13 positives
  Price Restrictions: 15 positives
  Unlimited/All-You-Can-Eat-License: 17 positives
38 clauses · 408 train / 51 val contracts


In [ ]:
# Helper: compute one row of the ablation table from a ModelArtifacts.
# use_per_clause=True applies per-clause tuned thresholds; otherwise uses the artifact's global threshold.

def evaluate_artifact(artifacts, use_per_clause=False):
    if use_per_clause:
        thresholds = tune_per_clause_thresholds(
            artifacts.val_logits, artifacts.val_labels, artifacts.id_to_clause,
        )
    else:
        # global probability threshold (same for every clause)
        thresholds = {name: artifacts.best_threshold for name in artifacts.id_to_clause.values()}
    metrics = compute_aggregate_metrics(
        artifacts.val_logits, artifacts.val_labels, thresholds,
        id_to_clause=artifacts.id_to_clause,
    )
    return metrics, thresholds

ablation_rows = []

def log_row(name, metrics, runtime_s):
    row = {
        'Variant':         name,
        'Micro-F1':        round(metrics['micro_f1'], 4),
        'Macro-F1':        round(metrics['macro_f1'], 4),
        'Micro-Precision': round(metrics['micro_precision'], 4),
        'Micro-Recall':    round(metrics['micro_recall'], 4),
        'Runtime (s)':     round(runtime_s, 1),
    }
    ablation_rows.append(row)
    print(f"\n→ {name}: micro-F1={row['Micro-F1']}, macro-F1={row['Macro-F1']} (took {row['Runtime (s)']}s)\n")

## V1 — Baseline (TF-IDF word 1-2 grams, 50k, LR C=1)

In [ ]:
t0 = time.time()
artifacts_v1 = train_tfidf_lr(train_df, val_df, id_to_clause)
runtime_v1 = time.time() - t0

metrics_v1, _ = evaluate_artifact(artifacts_v1, use_per_clause=False)
log_row('V1 — Baseline', metrics_v1, runtime_v1)

TF-IDF + LR → val micro_F1=0.7936, threshold=0.55

→ V1 — Baseline: micro-F1=0.7936, macro-F1=0.6318 (took 8.1s)



## V2 — Richer n-grams (word 1-3 + char 3-5, 150k features)

In [ ]:
t0 = time.time()
artifacts_v2 = train_tfidf_lr_v2(
    train_df, val_df, id_to_clause,
    ngram_word=(1, 3), ngram_char=(3, 5),
    max_features_word=100_000, max_features_char=50_000,
    tune_C=False, calibrate=False,
    artifact_name='V2 — word+char n-grams',
)
runtime_v2 = time.time() - t0

metrics_v2, _ = evaluate_artifact(artifacts_v2, use_per_clause=False)
log_row('V2 — word+char n-grams', metrics_v2, runtime_v2)

Features: 150,000 (word+char)
V2 — word+char n-grams → val micro_F1=0.7997, threshold=0.55

→ V2 — word+char n-grams: micro-F1=0.7997, macro-F1=0.6125 (took 44.3s)



## V3 — + Per-clause C tuning

In [ ]:
t0 = time.time()
artifacts_v3 = train_tfidf_lr_v2(
    train_df, val_df, id_to_clause,
    ngram_word=(1, 3), ngram_char=(3, 5),
    max_features_word=100_000, max_features_char=50_000,
    tune_C=True, calibrate=False,
    artifact_name='V3 — + per-clause C',
)
runtime_v3 = time.time() - t0

metrics_v3, _ = evaluate_artifact(artifacts_v3, use_per_clause=False)
log_row('V3 — + per-clause C', metrics_v3, runtime_v3)

Features: 150,000 (word+char)


/Users/saiashwin/BT5153/.claude/worktrees/crazy-tharp-7c055d/venv/lib/python3.13/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/Users/saiashwin/BT5153/.claude/worktrees/crazy-tharp-7c055d/venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:490: FitFailedWarning: 
3 fits failed out of a total of 9.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/saiashwin/BT5153/.claude/worktrees/crazy-tharp-7c055d/venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator

V3 — + per-clause C → val micro_F1=0.7833, threshold=0.55
  per-clause C distribution: {0.1: 15, 10.0: 12, 1.0: 10}

→ V3 — + per-clause C: micro-F1=0.7833, macro-F1=0.583 (took 147.5s)



## V4 — + Probability calibration (Platt scaling)

In [ ]:
t0 = time.time()
artifacts_v4 = train_tfidf_lr_v2(
    train_df, val_df, id_to_clause,
    ngram_word=(1, 3), ngram_char=(3, 5),
    max_features_word=100_000, max_features_char=50_000,
    tune_C=True, calibrate=True,
    artifact_name='V4 — + calibration',
)
runtime_v4 = time.time() - t0

metrics_v4, _ = evaluate_artifact(artifacts_v4, use_per_clause=False)
log_row('V4 — + calibration', metrics_v4, runtime_v4)

Features: 150,000 (word+char)


/Users/saiashwin/BT5153/.claude/worktrees/crazy-tharp-7c055d/venv/lib/python3.13/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/Users/saiashwin/BT5153/.claude/worktrees/crazy-tharp-7c055d/venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:490: FitFailedWarning: 
3 fits failed out of a total of 9.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/saiashwin/BT5153/.claude/worktrees/crazy-tharp-7c055d/venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator

ValueError: Requesting 3-fold cross-validation but provided less than 3 examples for at least one class.

## V5 — + Per-clause thresholds

Same model as V4; just swaps the global threshold for per-clause tuned thresholds at inference.

In [ ]:
t0 = time.time()
metrics_v5, thresholds_v5 = evaluate_artifact(artifacts_v4, use_per_clause=True)
runtime_v5 = time.time() - t0 + runtime_v4   # cumulative: reuses V4 training

log_row('V5 — + per-clause thresholds', metrics_v5, runtime_v5)

# Save thresholds for later — they'll live on the final checkpoint
print(f'Sample per-clause thresholds: {dict(list(thresholds_v5.items())[:5])}')

## V6 — + MiniLM embedding ensemble

Trains a second classifier on `sentence-transformers/all-MiniLM-L6-v2` embeddings, then averages its probabilities with V4's. First run downloads ~80 MB and caches embeddings; subsequent runs are instant.

In [ ]:
t0 = time.time()
artifacts_minilm = train_minilm_lr(
    train_df, val_df, id_to_clause,
    cache_path='checkpoints/minilm_embeddings.npz',
    tune_C=True,
)
runtime_minilm = time.time() - t0
print(f'MiniLM standalone micro-F1: {artifacts_minilm.val_metrics["micro_f1"]:.4f} (took {runtime_minilm:.1f}s)')

In [ ]:
# Ensemble: sweep weight_a on validation → average V4 (TF-IDF) and MiniLM probabilities → per-clause thresholds
t0 = time.time()
artifacts_v6 = ensemble_artifacts(
    artifacts_v4, artifacts_minilm,
    weight_a=None,           # sweep {0.1, 0.3, 0.5, 0.7, 0.9} and keep best
    artifact_name='V6 — TF-IDF + MiniLM ensemble',
)
metrics_v6, thresholds_v6 = evaluate_artifact(artifacts_v6, use_per_clause=True)
runtime_v6 = time.time() - t0 + runtime_v4 + runtime_minilm

log_row('V6 — + MiniLM ensemble', metrics_v6, runtime_v6)

## V7 — Early-fusion: TF-IDF ⊕ MiniLM features

Instead of averaging two models' probabilities (V6), concatenate TF-IDF sparse features with MiniLM dense embeddings into a single feature matrix and train one LR on the joint input. The LR learns the per-label weighting of each feature source — strictly more flexible than a single global `weight_a`.

In [ ]:
# V7: sweep minilm_scale on validation. Higher scale lets LR actually use the dense
# MiniLM dimensions despite L2 regularization treating every feature symmetrically.
t0 = time.time()

scale_candidates = [1.0, 10.0, 30.0, 100.0]
best = {'scale': None, 'f1': -1.0, 'artifacts': None, 'metrics': None, 'thresholds': None, 'used_pc': False}

for scale in scale_candidates:
    a = train_hybrid_features_lr(
        train_df, val_df, id_to_clause,
        ngram_word=(1, 3), ngram_char=(3, 5),
        max_features_word=100_000, max_features_char=50_000,
        cache_path='checkpoints/minilm_embeddings.npz',
        tune_C=False,
        minilm_scale=scale,
        artifact_name=f'V7 — Hybrid (scale={scale})',
        verbose=False,
    )
    m_g, _ = evaluate_artifact(a, use_per_clause=False)
    m_p, t_p = evaluate_artifact(a, use_per_clause=True)
    if m_p['micro_f1'] >= m_g['micro_f1']:
        m, t, used_pc = m_p, t_p, True
    else:
        m = m_g
        t = {name: a.best_threshold for name in a.id_to_clause.values()}
        used_pc = False
    print(f"  scale={scale:>5}: global={m_g['micro_f1']:.4f} per-clause={m_p['micro_f1']:.4f}  → using {'per-clause' if used_pc else 'global'}")
    if m['micro_f1'] > best['f1']:
        best = {'scale': scale, 'f1': m['micro_f1'], 'artifacts': a, 'metrics': m, 'thresholds': t, 'used_pc': used_pc}

runtime_v7 = time.time() - t0
artifacts_v7  = best['artifacts']
metrics_v7    = best['metrics']
thresholds_v7 = best['thresholds']

print(f"\nV7 best: minilm_scale={best['scale']}, threshold mode={'per-clause' if best['used_pc'] else 'global'}, micro-F1={best['f1']:.4f}")
log_row('V7 — Hybrid features (scaled)', metrics_v7, runtime_v7)

NameError: name 'train_hybrid_features_lr' is not defined

## Section 7 — Ablation Table

In [ ]:
ablation_df = pd.DataFrame(ablation_rows)
baseline_f1 = ablation_df.iloc[0]['Micro-F1']
ablation_df['Δ Micro-F1'] = (ablation_df['Micro-F1'] - baseline_f1).round(4)

# Save for the report
ablation_df.to_csv('figures/ablation_table.csv', index=False)
print('Saved figures/ablation_table.csv\n')
ablation_df


## Section 8 — Per-Clause F1 Chart (V1 vs V6)

In [ ]:
# Count train positives (needed by compute_per_clause_metrics)
def _count_train_positives(train_df, id_to_clause):
    name_to_id = {v: k for k, v in id_to_clause.items()}
    pos_count = {i: 0 for i in id_to_clause}
    for _, grp in train_df.groupby('contract_title'):
        for _, r in grp.iterrows():
            if r['has_answer'] and r['clause_type'] in name_to_id:
                pos_count[name_to_id[r['clause_type']]] += 1
    return pos_count

n_pos_train = _count_train_positives(train_df, id_to_clause)

# Pick the best artifact+thresholds by micro-F1 for the chart's "best" column
all_variants = {
    'V1': (artifacts_v1, {name: artifacts_v1.best_threshold for name in id_to_clause.values()}),
    'V2': (artifacts_v2, {name: artifacts_v2.best_threshold for name in id_to_clause.values()}),
    'V3': (artifacts_v3, {name: artifacts_v3.best_threshold for name in id_to_clause.values()}),
    'V4': (artifacts_v4, {name: artifacts_v4.best_threshold for name in id_to_clause.values()}),
    'V5': (artifacts_v4, thresholds_v5),
    'V6': (artifacts_v6, thresholds_v6),
    'V7': (artifacts_v7, thresholds_v7),
}
best_variant_key = ablation_df.iloc[ablation_df['Micro-F1'].idxmax()]['Variant'].split(' ')[0]
best_artifact_for_chart, best_thresholds_for_chart = all_variants[best_variant_key]

thr_v1 = {name: artifacts_v1.best_threshold for name in id_to_clause.values()}

pc_v1 = compute_per_clause_metrics(
    artifacts_v1.val_logits, artifacts_v1.val_labels, thr_v1, id_to_clause, n_pos_train,
)
pc_best = compute_per_clause_metrics(
    best_artifact_for_chart.val_logits, best_artifact_for_chart.val_labels,
    best_thresholds_for_chart, id_to_clause, n_pos_train,
)

merged = pc_v1.set_index('clause_type')[['f1']].rename(columns={'f1': 'V1'}).join(
    pc_best.set_index('clause_type')[['f1']].rename(columns={'f1': best_variant_key}),
).sort_values('V1').reset_index()

fig, ax = plt.subplots(figsize=(12, max(6, 0.28 * len(merged))))
y = np.arange(len(merged))
ax.barh(y - 0.2, merged['V1'],             height=0.4, label='V1 baseline', color='#b0b0b0')
ax.barh(y + 0.2, merged[best_variant_key], height=0.4, label=f'{best_variant_key} (best)', color='#2a9d8f')
ax.set_yticks(y)
ax.set_yticklabels(merged['clause_type'], fontsize=9)
ax.set_xlabel('F1')
ax.set_xlim(0, 1.0)
ax.set_title(f'Per-clause F1 — V1 baseline vs {best_variant_key} best')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/per_clause_f1_chart.png', dpi=160, bbox_inches='tight')
print(f'Saved figures/per_clause_f1_chart.png (V1 vs {best_variant_key})')
plt.show()


## Section 9 — Export Best Checkpoint for the Webapp

In [ ]:
import joblib

# Pick the best artifact by micro-F1
all_variants = {
    'V1': (artifacts_v1, None),
    'V2': (artifacts_v2, None),
    'V3': (artifacts_v3, None),
    'V4': (artifacts_v4, None),
    'V5': (artifacts_v4, thresholds_v5),
    'V6': (artifacts_v6, thresholds_v6),
    'V7': (artifacts_v7, thresholds_v7),
}
best_name = ablation_df.iloc[ablation_df['Micro-F1'].idxmax()]['Variant'].split(' ')[0]
best_artifact, best_thresholds = all_variants[best_name]

best_threshold_to_save = best_thresholds if best_thresholds is not None else best_artifact.best_threshold

ckpt_path = 'checkpoints/tfidf_lr_v2_artifacts.joblib'
joblib.dump({
    'model':          best_artifact.model,
    'best_threshold': best_threshold_to_save,
    'id_to_clause':   best_artifact.id_to_clause,
    'variant':        best_name,
    'val_metrics':    ablation_df[ablation_df['Variant'].str.startswith(best_name)].iloc[0].to_dict(),
}, ckpt_path)

print(f'Best variant: {best_name}')
print(f'Saved checkpoint → {ckpt_path}')
print('\nTo use this in the webapp:')
print('  cp checkpoints/tfidf_lr_artifacts.joblib checkpoints/tfidf_lr_artifacts.joblib.bak')
print(f'  cp {ckpt_path} checkpoints/tfidf_lr_artifacts.joblib')
print('  streamlit run webapp/app.py')
